In [ ]:
# Install required packages if needed
%pip -q install openpyxl seaborn

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from google.colab import files

# --------------------------------------------------
# STEP 1: UPLOAD FILE
# --------------------------------------------------

print("Please upload your raw data Excel file.")
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one Excel file.")

file_path = next(iter(uploaded))
df = pd.read_excel(file_path)

df.columns = df.columns.astype(str).str.strip()

print(f"Loaded {len(df):,} rows of data.")

# --------------------------------------------------
# STEP 2: DEFINE METRICS
# --------------------------------------------------

cmj_cols = [
    'Concentric Duration',
    'Concentric Mean Force',
    'Concentric Mean Power / BM',
    'Concentric Peak Force',
    'Eccentric:Concentric Mean Force Ratio',
    'Eccentric Deceleration Phase Duration',
    'Eccentric Duration',
    'Eccentric Deceleration Mean Force',
    'Eccentric Peak Power',
    'Eccentric Peak Power / BM',
    'Eccentric Peak Velocity'
]

trackman_cols = [
    'RelSpeed',
    'SpinRate',
    'SpinAxis',
    'VertBreak',
    'InducedVertBreak',
    'HorzBreak'
]

all_analysis_cols = cmj_cols + trackman_cols

required_cols = [
    'Date',
    'PitcherID',
    'TaggedPitchType',
    *all_analysis_cols
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(
        "The uploaded spreadsheet is missing these required columns:\n"
        + "\n".join(missing_cols)
    )

if df.columns.duplicated().any():
    raise ValueError(
        "The spreadsheet contains duplicate column names after "
        "removing leading and trailing spaces."
    )

# --------------------------------------------------
# STEP 3: CLEAN DATES AND IDENTIFIERS
# --------------------------------------------------

df['Date'] = pd.to_datetime(
    df['Date'],
    errors='coerce'
).dt.normalize()

df['PitcherID'] = (
    df['PitcherID']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

df['TaggedPitchType'] = (
    df['TaggedPitchType']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

# Exclude records that cannot be assigned to a
# pitcher, date, or pitch type.

rows_before = len(df)

df = df.dropna(
    subset=['Date', 'PitcherID', 'TaggedPitchType']
).copy()

print(
    f"Excluded {rows_before - len(df):,} rows "
    "with missing pitcher IDs, dates, or pitch types."
)

if df.empty:
    raise ValueError(
        "No usable rows remain after checking "
        "pitcher IDs, dates, and pitch types."
    )

# --------------------------------------------------
# STEP 4: CONVERT METRICS TO NUMERIC
# --------------------------------------------------

for col in all_analysis_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --------------------------------------------------
# STEP 5: FORWARD-FILL CMJ DATA WITHIN SESSIONS
# --------------------------------------------------

# This preserves the original script's approach.
#
# CMJ values are carried forward only within the
# same pitcher and calendar date.
#
# IMPORTANT:
# This assumes the spreadsheet's row order reflects
# the intended order of measurements within a session.
# Forward-filling is not appropriate if the row order
# is arbitrary or if different CMJ trials should be
# matched to different pitches.

print("Forward-filling CMJ data within sessions...")

df_filled = df.copy()

df_filled[cmj_cols] = (
    df_filled
    .groupby(
        ['Date', 'PitcherID'],
        sort=False
    )[cmj_cols]
    .ffill()
)

# --------------------------------------------------
# STEP 6: REMOVE ROWS WITH MISSING ANALYSIS METRICS
# --------------------------------------------------

rows_before_cleaning = len(df_filled)

df_clean = (
    df_filled
    .dropna(subset=all_analysis_cols)
    .copy()
)

print(
    f"Removed {rows_before_cleaning - len(df_clean):,} "
    "rows with missing analysis metrics."
)

print(
    f"Rows remaining for correlation analysis: "
    f"{len(df_clean):,}"
)

if df_clean.empty:
    raise ValueError(
        "No rows remain after removing missing "
        "analysis metrics."
    )

# --------------------------------------------------
# STEP 7: CALCULATE FASTBALL PERCENTAGE
# --------------------------------------------------

# Use the pitching records BEFORE removing rows
# with missing CMJ or TrackMan metrics.
#
# This prevents missing measurements from changing
# a pitcher's fastball percentage.

pitcher_totals = (
    df
    .groupby('PitcherID')
    .size()
)

fastball_counts = (
    df['TaggedPitchType']
    .eq('Fastball')
    .groupby(df['PitcherID'])
    .sum()
)

fastball_pct = fastball_counts.div(pitcher_totals)

# Every pitcher is included, including pitchers
# with zero fastballs.

fb_pitchers = fastball_pct[
    fastball_pct >= 0.50
].index

os_pitchers = fastball_pct[
    fastball_pct < 0.50
].index

print(
    f"Fastball-dominant pitchers: "
    f"{len(fb_pitchers)}"
)

print(
    f"Offspeed-dominant pitchers: "
    f"{len(os_pitchers)}"
)

# --------------------------------------------------
# STEP 8: ASSIGN PITCHER GROUP
# --------------------------------------------------

df_clean['PitcherType'] = np.where(
    df_clean['PitcherID'].isin(fb_pitchers),
    'Fastball',
    'Offspeed'
)

# --------------------------------------------------
# STEP 9: CREATE OUTPUT FOLDER
# --------------------------------------------------

output_dir = Path('/content/cmj_pitching_heatmaps')
output_dir.mkdir(parents=True, exist_ok=True)

created_files = []

# --------------------------------------------------
# STEP 10: DEFINE HEATMAP FUNCTION
# --------------------------------------------------

def create_correlation_matrix(data, title, output_name):
    """
    Generate a correlation matrix heatmap and
    export its correlation coefficients to Excel.

    Correlations are descriptive. Multiple pitches
    may share the same CMJ measurement.
    """

    if len(data) < 10:
        print(
            f"Skipped {title}: "
            f"fewer than 10 pitches (n={len(data)})."
        )
        return None

    corr_data = data[all_analysis_cols].copy()

    corr_matrix = corr_data.corr(method='pearson')

    if corr_matrix.isna().all().all():
        print(
            f"Skipped {title}: "
            "no valid correlations could be calculated."
        )
        return None

    # Hide the upper triangle of the matrix.

    mask = np.triu(
        np.ones_like(corr_matrix, dtype=bool)
    )

    fig, ax = plt.subplots(figsize=(16, 14))

    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={'shrink': 0.8},
        vmin=-1,
        vmax=1,
        annot_kws={'size': 8},
        ax=ax
    )

    ax.set_title(
        f'{title}\n(n={len(data)} pitches)',
        fontsize=16,
        pad=20
    )

    fig.tight_layout()

    # Save the heatmap image.

    image_path = output_dir / f'{output_name}.png'

    fig.savefig(
        image_path,
        dpi=300,
        bbox_inches='tight'
    )

    created_files.append(image_path)

    print(f"Saved: {image_path.name}")

    # Save the numerical correlation matrix.

    excel_path = output_dir / f'{output_name}.xlsx'

    corr_matrix.to_excel(excel_path)

    created_files.append(excel_path)

    print(f"Saved: {excel_path.name}")

    plt.show()
    plt.close(fig)

    return corr_matrix

# --------------------------------------------------
# STEP 11: DEFINE PITCHER GROUPS AND PITCH TYPES
# --------------------------------------------------

groups = {
    'FastballPitchers': {
        'label': 'Fastball Pitchers',
        'pitcher_type': 'Fastball'
    },
    'OffspeedPitchers': {
        'label': 'Offspeed Pitchers',
        'pitcher_type': 'Offspeed'
    }
}

pitch_types = [
    'Fastball',
    'ChangeUp',
    'Slider',
    'Curveball'
]

# --------------------------------------------------
# STEP 12: GENERATE CORRELATION MATRICES
# --------------------------------------------------

print("\n" + "=" * 60)
print("GENERATING CORRELATION MATRICES")
print("=" * 60)

for group_name, group_info in groups.items():

    for pitch_type in pitch_types:

        subset = df_clean.loc[
            (df_clean['PitcherType'] == group_info['pitcher_type'])
            &
            (df_clean['TaggedPitchType'] == pitch_type)
        ].copy()

        title = (
            f"{group_info['label']}: "
            f"{pitch_type} Correlations"
        )

        output_name = (
            f"{group_name}_{pitch_type}"
        )

        create_correlation_matrix(
            subset,
            title,
            output_name
        )

# --------------------------------------------------
# STEP 13: REPORT ACTUAL OUTPUTS
# --------------------------------------------------

image_files = [
    path for path in created_files
    if path.suffix == '.png'
]

excel_files = [
    path for path in created_files
    if path.suffix == '.xlsx'
]

print("\n" + "=" * 60)
print("CORRELATION MATRIX GENERATION COMPLETE")
print("=" * 60)

print(f"Heatmap images created: {len(image_files)}")
print(f"Excel correlation matrices created: {len(excel_files)}")

if created_files:
    print("\nFiles saved in:")
    print(output_dir)
else:
    print(
        "\nNo files were created. "
        "Check the available pitch counts and metrics."
    )

# Do not upload real-data outputs to a public GitHub repository.